In [1]:
!nvidia-smi

Wed Nov 25 11:47:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [3]:
%%writefile test.cu

#include <stdio.h>
#include <cuda_runtime.h>

__global__ void helloCUDA()
{
    printf("Hello from CUDA GPU! Thread ID: %d\n",
           threadIdx.x);
}

int main()
{
    helloCUDA<<<1, 1>>>();

    cudaDeviceSynchronize();

    printf("CUDA program executed successfully!\n");

    return 0;
}

Writing test.cu


In [5]:
!nvcc test.cu -o test

In [6]:
!./test

Hello from CUDA GPU! Thread ID: 0
CUDA program executed successfully!


In [8]:
%%writefile reduction.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <chrono>

#define N (1 << 24)
#define THREADS 256

// CPU Reduction
float cpuReduction(float *a, int n)
{
    float sum = 0.0f;

    for (int i = 0; i < n; i++)
        sum += a[i];

    return sum;
}

// Naive CUDA Reduction
__global__ void naiveReduction(
    float *input, float *output, int n)
{
    __shared__ float sdata[THREADS];

    unsigned int tid = threadIdx.x;
    unsigned int i = blockIdx.x * blockDim.x + tid;

    sdata[tid] = (i < n) ? input[i] : 0.0f;

    __syncthreads();

    // Interleaved addressing
    for (unsigned int s = 1; s < blockDim.x; s *= 2)
    {
        if (tid % (2 * s) == 0)
        {
            sdata[tid] += sdata[tid + s];
        }

        __syncthreads();
    }

    if (tid == 0)
        output[blockIdx.x] = sdata[0];
}

int main()
{
    float *h_input = (float*)malloc(N * sizeof(float));

    for (int i = 0; i < N; i++)
        h_input[i] = 1.0f;

    // CPU execution
    auto start = std::chrono::high_resolution_clock::now();

    float cpuSum = cpuReduction(h_input, N);

    auto stop = std::chrono::high_resolution_clock::now();

    double cpuTime =
        std::chrono::duration<double, std::milli>(
            stop - start).count();

    // GPU memory allocation
    float *d_input, *d_partial, *h_partial;

    int blocks = (N + THREADS - 1) / THREADS;

    cudaMalloc(&d_input, N * sizeof(float));
    cudaMalloc(&d_partial, blocks * sizeof(float));

    h_partial = (float*)malloc(blocks * sizeof(float));

    cudaMemcpy(d_input, h_input,
        N * sizeof(float), cudaMemcpyHostToDevice);

    // Warm-up
    naiveReduction<<<blocks, THREADS>>>(
        d_input, d_partial, N);

    cudaDeviceSynchronize();

    // CUDA event timing
    cudaEvent_t startEvent, stopEvent;

    cudaEventCreate(&startEvent);
    cudaEventCreate(&stopEvent);

    cudaEventRecord(startEvent);

    for (int r = 0; r < 10; r++)
    {
        naiveReduction<<<blocks, THREADS>>>(
            d_input, d_partial, N);
    }

    cudaEventRecord(stopEvent);
    cudaEventSynchronize(stopEvent);

    float totalTime;

    cudaEventElapsedTime(
        &totalTime, startEvent, stopEvent);

    float gpuTime = totalTime / 10.0f;

    cudaMemcpy(h_partial, d_partial,
        blocks * sizeof(float), cudaMemcpyDeviceToHost);

    float gpuSum = 0.0f;

    for (int i = 0; i < blocks; i++)
        gpuSum += h_partial[i];

    printf("\n----- RESULTS -----\n");

    printf("CPU Sum       : %.0f\n", cpuSum);
    printf("Naive GPU Sum : %.0f\n", gpuSum);

    printf("CPU Time      : %.4f ms\n", cpuTime);
    printf("Naive GPU Kernel Time: %.4f ms\n", gpuTime);

    printf("Results Match : %s\n",
        (cpuSum == gpuSum) ? "YES" : "NO");

    cudaFree(d_input);
    cudaFree(d_partial);

    free(h_input);
    free(h_partial);

    cudaEventDestroy(startEvent);
    cudaEventDestroy(stopEvent);

    return 0;
}


Writing reduction.cu


In [9]:
!nvcc reduction.cu -o reduction

In [10]:
!./reduction


----- RESULTS -----
CPU Sum       : 16777216
Naive GPU Sum : 16777216
CPU Time      : 49.8654 ms
Naive GPU Kernel Time: 1.8882 ms
Results Match : YES


In [11]:
%%writefile optimized.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <chrono>

#define N (1 << 24)
#define THREADS 256

// CPU reduction
float cpuReduction(float *a, int n) {
    float sum = 0.0f;

    for (int i = 0; i < n; i++) {
        sum += a[i];
    }

    return sum;
}

// Optimised CUDA reduction
__global__ void optimizedReduction(float *input, float *output, int n) {

    __shared__ float sdata[THREADS];

    unsigned int tid = threadIdx.x;

    // Each thread adds two elements during loading
    unsigned int i = blockIdx.x * (blockDim.x * 2) + tid;

    float sum = 0.0f;

    if (i < n)
        sum += input[i];

    if (i + blockDim.x < n)
        sum += input[i + blockDim.x];

    sdata[tid] = sum;
    __syncthreads();

    // Sequential addressing using shared memory
    for (unsigned int s = blockDim.x / 2; s >= 32; s >>= 1) {
        if (tid < s) {
            sdata[tid] += sdata[tid + s];
        }
        __syncthreads();
    }

    // Warp shuffle reduction
    if (tid < 32) {
        float value = sdata[tid];

        for (int offset = 16; offset > 0; offset >>= 1) {
            value += __shfl_down_sync(0xffffffff, value, offset);
        }

        if (tid == 0) {
            output[blockIdx.x] = value;
        }
    }
}

// Launch multiple stages until one final sum remains
float* launchOptimized(float *d_input,
                       float *d_temp1,
                       float *d_temp2,
                       int n) {

    float *current = d_input;
    float *next = d_temp1;

    while (n > 1) {

        int blocks = (n + THREADS * 2 - 1) / (THREADS * 2);

        optimizedReduction<<<blocks, THREADS>>>(current, next, n);

        n = blocks;
        current = next;

        next = (next == d_temp1) ? d_temp2 : d_temp1;
    }

    return current;
}

int main() {

    float *h_input = (float*)malloc(N * sizeof(float));

    for (int i = 0; i < N; i++) {
        h_input[i] = 1.0f;
    }

    // CPU calculation and timing
    auto cpuStart = std::chrono::high_resolution_clock::now();

    float cpuSum = cpuReduction(h_input, N);

    auto cpuEnd = std::chrono::high_resolution_clock::now();

    double cpuTime =
        std::chrono::duration<double, std::milli>(
            cpuEnd - cpuStart).count();

    // Allocate GPU memory
    float *d_input, *d_temp1, *d_temp2;

    int maxBlocks = (N + THREADS * 2 - 1) / (THREADS * 2);

    cudaMalloc(&d_input, N * sizeof(float));
    cudaMalloc(&d_temp1, maxBlocks * sizeof(float));
    cudaMalloc(&d_temp2, maxBlocks * sizeof(float));

    cudaMemcpy(d_input, h_input, N * sizeof(float),
               cudaMemcpyHostToDevice);

    // Warm-up
    float *d_result = launchOptimized(
        d_input, d_temp1, d_temp2, N);

    cudaDeviceSynchronize();

    // CUDA event timing
    cudaEvent_t start, stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    int repetitions = 10;

    cudaEventRecord(start);

    for (int r = 0; r < repetitions; r++) {
        d_result = launchOptimized(
            d_input, d_temp1, d_temp2, N);
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float totalTime = 0.0f;

    cudaEventElapsedTime(&totalTime, start, stop);

    float gpuTime = totalTime / repetitions;

    // Copy final GPU result to CPU
    cudaMemcpy(&d_result[0], &d_result[0], sizeof(float),
               cudaMemcpyDeviceToDevice);

    float gpuSum;

    cudaMemcpy(&gpuSum, d_result, sizeof(float),
               cudaMemcpyDeviceToHost);

    // Display results
    printf("\n----- OPTIMISED RESULTS -----\n");

    printf("CPU Sum             : %.0f\n", cpuSum);
    printf("Optimised GPU Sum   : %.0f\n", gpuSum);

    printf("CPU Time            : %.4f ms\n", cpuTime);
    printf("Optimised GPU Time  : %.4f ms\n", gpuTime);

    printf("Speedup             : %.2f times\n",
           cpuTime / gpuTime);

    printf("Results Match       : %s\n",
           (cpuSum == gpuSum) ? "YES" : "NO");

    // Free memory
    cudaFree(d_input);
    cudaFree(d_temp1);
    cudaFree(d_temp2);

    free(h_input);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}

Writing optimized.cu


In [13]:
!nvcc optimized.cu -o optimized

In [14]:
!./optimized


----- OPTIMISED RESULTS -----
CPU Sum             : 16777216
Optimised GPU Sum   : 16777216
CPU Time            : 54.3459 ms
Optimised GPU Time  : 0.4811 ms
Speedup             : 112.97 times
Results Match       : YES


In [3]:

%%writefile naive_corrected.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <chrono>

#define N (1 << 24)
#define THREADS 256

// CPU reduction
float cpuReduction(float *a, int n) {
    float sum = 0.0f;

    for (int i = 0; i < n; i++) {
        sum += a[i];
    }

    return sum;
}

// Naive CUDA reduction using interleaved addressing
__global__ void naiveReduction(float *input, float *output, int n) {

    __shared__ float sdata[THREADS];

    unsigned int tid = threadIdx.x;
    unsigned int i = blockIdx.x * blockDim.x + tid;

    sdata[tid] = (i < n) ? input[i] : 0.0f;

    __syncthreads();

    for (unsigned int s = 1; s < blockDim.x; s *= 2) {

        if (tid % (2 * s) == 0) {
            sdata[tid] += sdata[tid + s];
        }

        __syncthreads();
    }

    if (tid == 0) {
        output[blockIdx.x] = sdata[0];
    }
}

// Repeat reduction stages until one sum remains
float* launchNaive(float *d_input,
                   float *d_temp1,
                   float *d_temp2,
                   int n) {

    float *current = d_input;
    float *next = d_temp1;

    while (n > 1) {

        int blocks = (n + THREADS - 1) / THREADS;

        naiveReduction<<<blocks, THREADS>>>(current, next, n);

        n = blocks;
        current = next;

        next = (next == d_temp1) ? d_temp2 : d_temp1;
    }

    return current;
}

int main() {

    float *h_input = (float*)malloc(N * sizeof(float));

    for (int i = 0; i < N; i++) {
        h_input[i] = 1.0f;
    }

    // CPU calculation and timing
    auto cpuStart = std::chrono::high_resolution_clock::now();

    float cpuSum = cpuReduction(h_input, N);

    auto cpuEnd = std::chrono::high_resolution_clock::now();

    double cpuTime =
        std::chrono::duration<double, std::milli>(
            cpuEnd - cpuStart).count();

    // Allocate GPU memory
    float *d_input, *d_temp1, *d_temp2;

    int maxBlocks = (N + THREADS - 1) / THREADS;

    cudaMalloc(&d_input, N * sizeof(float));
    cudaMalloc(&d_temp1, maxBlocks * sizeof(float));
    cudaMalloc(&d_temp2, maxBlocks * sizeof(float));

    cudaMemcpy(d_input, h_input, N * sizeof(float),
               cudaMemcpyHostToDevice);

    // Warm-up
    float *d_result = launchNaive(
        d_input, d_temp1, d_temp2, N);

    cudaDeviceSynchronize();

    // CUDA event timing
    cudaEvent_t start, stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    int repetitions = 10;

    cudaEventRecord(start);

    for (int r = 0; r < repetitions; r++) {
        d_result = launchNaive(
            d_input, d_temp1, d_temp2, N);
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float totalTime = 0.0f;

    cudaEventElapsedTime(&totalTime, start, stop);

    float gpuTime = totalTime / repetitions;

    // Copy final GPU result to CPU
    float gpuSum;

    cudaMemcpy(&gpuSum, d_result, sizeof(float),
               cudaMemcpyDeviceToHost);

    // Display results
    printf("\n----- CORRECTED NAIVE RESULTS -----\n");

    printf("CPU Sum          : %.0f\n", cpuSum);
    printf("Naive GPU Sum    : %.0f\n", gpuSum);

    printf("CPU Time         : %.4f ms\n", cpuTime);
    printf("Naive GPU Time   : %.4f ms\n", gpuTime);

    printf("Speedup          : %.2f times\n",
           cpuTime / gpuTime);

    printf("Results Match    : %s\n",
           (cpuSum == gpuSum) ? "YES" : "NO");

    // Free memory
    cudaFree(d_input);
    cudaFree(d_temp1);
    cudaFree(d_temp2);

    free(h_input);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}


Writing naive_corrected.cu


In [3]:
!nvcc naive_corrected.cu -o naive_corrected

/bin/bash: line 1: nvcc: command not found


In [1]:
!nvidia-smi

Wed Nov 25 12:56:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [4]:
!nvcc naive_corrected.cu -o naive_corrected

In [5]:
!./naive_corrected


----- CORRECTED NAIVE RESULTS -----
CPU Sum          : 16777216
Naive GPU Sum    : 16777216
CPU Time         : 52.1049 ms
Naive GPU Time   : 1.5740 ms
Speedup          : 33.10 times
Results Match    : YES


In [7]:

%%writefile block_size_test.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N (1 << 24)

__global__ void optimizedReduction(
    float *input, float *output, int n) {

    extern __shared__ float sdata[];

    unsigned int tid = threadIdx.x;
    unsigned int blockSize = blockDim.x;

    unsigned int i =
        blockIdx.x * (blockSize * 2) + tid;

    float sum = 0.0f;

    if (i < n)
        sum += input[i];

    if (i + blockSize < n)
        sum += input[i + blockSize];

    sdata[tid] = sum;
    __syncthreads();

    // Sequential addressing
    for (unsigned int s = blockSize / 2; s >= 32; s >>= 1) {
        if (tid < s)
            sdata[tid] += sdata[tid + s];

        __syncthreads();
    }

    // Warp shuffle
    if (tid < 32) {
        float value = sdata[tid];

        for (int offset = 16; offset > 0; offset >>= 1) {
            value += __shfl_down_sync(
                0xffffffff, value, offset);
        }

        if (tid == 0)
            output[blockIdx.x] = value;
    }
}

// Complete reduction on GPU
float* runReduction(
    float *d_input,
    float *d_temp1,
    float *d_temp2,
    int n,
    int threads) {

    float *current = d_input;
    float *next = d_temp1;

    while (n > 1) {

        int blocks = (n + threads * 2 - 1)
                     / (threads * 2);

        size_t sharedMemory = threads * sizeof(float);

        optimizedReduction<<<blocks, threads, sharedMemory>>>(
            current, next, n);

        n = blocks;
        current = next;

        next = (next == d_temp1) ? d_temp2 : d_temp1;
    }

    return current;
}

int main() {

    float *h_input =
        (float*)malloc(N * sizeof(float));

    for (int i = 0; i < N; i++)
        h_input[i] = 1.0f;

    float *d_input, *d_temp1, *d_temp2;

    cudaMalloc(&d_input, N * sizeof(float));

    cudaMemcpy(d_input, h_input,
               N * sizeof(float),
               cudaMemcpyHostToDevice);

    int threadSizes[] = {128, 256, 512};
    int repetitions = 10;

    printf("\n===== BLOCK SIZE COMPARISON =====\n");
    printf("Input size: %d elements\n", N);
    printf("GPU: NVIDIA T4 (Colab)\n\n");

    printf("Threads/Block     GPU Time (ms)     GPU Sum\n");

    for (int k = 0; k < 3; k++) {

        int threads = threadSizes[k];

        int maxBlocks =
            (N + threads * 2 - 1) / (threads * 2);

        cudaMalloc(&d_temp1,
                   maxBlocks * sizeof(float));

        cudaMalloc(&d_temp2,
                   maxBlocks * sizeof(float));

        // Warm-up
        float *d_result = runReduction(
            d_input, d_temp1, d_temp2, N, threads);

        cudaDeviceSynchronize();

        cudaEvent_t start, stop;
        cudaEventCreate(&start);
        cudaEventCreate(&stop);

        cudaEventRecord(start);

        for (int r = 0; r < repetitions; r++) {
            d_result = runReduction(
                d_input, d_temp1, d_temp2, N, threads);
        }

        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        float totalTime = 0.0f;

        cudaEventElapsedTime(
            &totalTime, start, stop);

        float averageTime = totalTime / repetitions;

        float gpuSum;

        cudaMemcpy(&gpuSum, d_result,
                   sizeof(float),
                   cudaMemcpyDeviceToHost);

        printf("%-18d %-18.4f %.0f\n",
               threads, averageTime, gpuSum);

        cudaEventDestroy(start);
        cudaEventDestroy(stop);

        cudaFree(d_temp1);
        cudaFree(d_temp2);
    }

    cudaFree(d_input);
    free(h_input);

    return 0;
}


Overwriting block_size_test.cu


In [8]:
!nvcc block_size_test.cu -o block_size_test

In [9]:
!./block_size_test


===== BLOCK SIZE COMPARISON =====
Input size: 16777216 elements
GPU: NVIDIA T4 (Colab)

Threads/Block     GPU Time (ms)     GPU Sum
128                0.3203             16777216
256                0.3596             16777216
512                0.4350             16777216


In [10]:
from google.colab import files
files.download('block_size_test.cu')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from google.colab import files
files.download('optimized.cu')

FileNotFoundError: Cannot find file: optimized.cu

In [12]:
!ls -lh *.cu

-rw-r--r-- 1 root root 3.5K Nov 25 13:04 block_size_test.cu
-rw-r--r-- 1 root root 3.5K Nov 25 12:57 naive_corrected.cu


In [13]:
from google.colab import files
files.download('naive_corrected.cu')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>